# Daily Assessment Export

This notebook transforms the wide_df from `00_setup.ipynb` into the REDCap daily assessment upload format.

**Input:** `output/intermediate/wide_df.parquet`, `output/intermediate/static_df.parquet`

**Output:** `output/final/daily_assessment_upload.csv`


## Setup


In [ ]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Paths
INTERMEDIATE_DIR = Path('../output/intermediate')
FINAL_DIR = Path('../output/final')
CONFIG_DIR = Path('../config')

# Ensure output directory exists
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete")


## Load Data and Mapping Config


In [ ]:
# Load wide_df and static_df
wide_df = pd.read_parquet(INTERMEDIATE_DIR / 'wide_df.parquet')
static_df = pd.read_parquet(INTERMEDIATE_DIR / 'static_df.parquet')

# Load mapping config
with open(CONFIG_DIR / 'daily_assessment_mapping.yaml', 'r') as f:
    mapping_config = yaml.safe_load(f)

print(f"Loaded wide_df: {wide_df.shape}")
print(f"Loaded static_df: {static_df.shape}")
print(f"Days to process: {mapping_config['days_to_process']}")


In [ ]:
# Preview data
print("Wide DF columns:", wide_df.columns.tolist()[:20], "...")
print(f"\nUnique patients: {wide_df['patient_id'].nunique()}")
print(f"Day range: {wide_df['day'].min()} to {wide_df['day'].max()}")


## Helper Functions


In [ ]:
def get_day_suffix(day: int) -> str:
    """Convert day number to REDCap suffix."""
    return mapping_config['day_suffixes'][day]


def get_closest_to_8am(df: pd.DataFrame, value_col: str) -> float:
    """
    Get the value closest to 8am from a day's data.
    
    Parameters:
        df: DataFrame filtered to a single day
        value_col: Column name to extract value from
    
    Returns:
        Value closest to 8am, or None if no valid data
    """
    if df.empty or value_col not in df.columns:
        return None
    
    # Filter to rows with non-null values for this column
    valid_df = df[df[value_col].notna()].copy()
    
    if valid_df.empty:
        return None
    
    # Calculate time distance from 8am
    valid_df['hour'] = pd.to_datetime(valid_df['event_dttm']).dt.hour
    valid_df['minute'] = pd.to_datetime(valid_df['event_dttm']).dt.minute
    valid_df['minutes_from_8am'] = abs((valid_df['hour'] * 60 + valid_df['minute']) - 8 * 60)
    
    # Get row closest to 8am
    closest_idx = valid_df['minutes_from_8am'].idxmin()
    return valid_df.loc[closest_idx, value_col]


def get_extreme_value(df: pd.DataFrame, value_col: str, aggregation: str) -> float:
    """
    Get min/max value from a day's data.
    
    Parameters:
        df: DataFrame filtered to a single day
        value_col: Column name to extract value from
        aggregation: 'min' or 'max'
    
    Returns:
        Aggregated value, or None if no valid data
    """
    if df.empty or value_col not in df.columns:
        return None
    
    values = df[value_col].dropna()
    
    if values.empty:
        return None
    
    if aggregation == 'min':
        return values.min()
    elif aggregation == 'max':
        return values.max()
    else:
        return None


print("Helper functions defined")


## Process Each Patient

In [ ]:
def process_patient(patient_id: str, record_id: str, patient_wide: pd.DataFrame) -> dict:
    """
    Process a single patient's data into REDCap daily assessment format.
    
    Parameters:
        patient_id: Patient identifier
        record_id: Record identifier from clinical trials table
        patient_wide: Wide DataFrame filtered to this patient
    
    Returns:
        Dictionary with all REDCap columns for this patient
    """
    # Use actual record_id, fallback to patient_id if None or NaN
    final_record_id = record_id if record_id and not pd.isna(record_id) else patient_id

    result = {
        'record_id': final_record_id,
        'redcap_event_name': mapping_config['output']['redcap_event_name']
    }

    days_to_process = mapping_config['days_to_process']

    for day in days_to_process:
        day_suffix = get_day_suffix(day)

        # Filter to this day
        day_data = patient_wide[patient_wide['day'] == day]

        # ----- Vitals closest to 8am -----
        for clif_col, config in mapping_config['vitals_8am'].items():
            redcap_col = f"{config['redcap_col']}_{day_suffix}"
            value = get_closest_to_8am(day_data, clif_col)
            result[redcap_col] = value

        # ----- Vitals extreme values -----
        for clif_col, config in mapping_config['vitals_extreme'].items():
            redcap_col = f"{config['redcap_col']}_{day_suffix}"
            value = get_extreme_value(day_data, clif_col, config['aggregation'])
            result[redcap_col] = value

        # ----- Labs closest to 8am -----
        for clif_col, config in mapping_config['labs_8am'].items():
            redcap_col = f"{config['redcap_col']}_{day_suffix}"
            nc_col = f"{config['redcap_nc_col']}_{day_suffix}"

            value = get_closest_to_8am(day_data, clif_col)
            result[redcap_col] = value
            # Set "not collected" flag if no value
            result[nc_col] = 1 if pd.isna(value) else None

        # ----- Labs highest values -----
        for clif_col, config in mapping_config['labs_highest'].items():
            redcap_col = f"{config['redcap_col']}_{day_suffix}"
            nc_col = f"{config['redcap_nc_col']}_{day_suffix}"

            value = get_extreme_value(day_data, clif_col, config['aggregation'])
            result[redcap_col] = value
            result[nc_col] = 1 if pd.isna(value) else None

        # ----- Labs NOT in CLIF (always not collected) -----
        for config in mapping_config['labs_not_in_clif']:
            redcap_col = f"{config['redcap_col']}_{day_suffix}"
            nc_col = f"{config['redcap_nc_col']}_{day_suffix}"
            result[redcap_col] = None
            result[nc_col] = 1

    return result

In [ ]:
# Process all patients
patient_ids = wide_df['patient_id'].unique()
print(f"Processing {len(patient_ids)} patients...")

all_results = []

for i, patient_id in enumerate(patient_ids):
    patient_wide = wide_df[wide_df['patient_id'] == patient_id]

    # Get record_id from the data
    record_id = patient_wide['record_id'].iloc[0] 

    # Process with both ids
    result = process_patient(patient_id, record_id, patient_wide)
    all_results.append(result)

    if (i + 1) % 10 == 0 or (i + 1) == len(patient_ids):
        print(f"  Processed {i + 1}/{len(patient_ids)} patients")



In [ ]:
all_results

## Create Output DataFrame

In [ ]:
# Convert to DataFrame
output_df = pd.DataFrame(all_results)

# Reorder columns to match expected format
data_cols = [c for c in output_df.columns if c not in ['record_id', 'redcap_event_name']]

def sort_key(col):
    parts = col.rsplit('_', 1)
    if len(parts) == 2:
        day_suffix = parts[1]
        if day_suffix == 'm2': day_num = -2
        elif day_suffix == 'm1': day_num = -1
        else:
            try: day_num = int(day_suffix)
            except: day_num = 99
        return (day_num, parts[0])
    return (99, col)

sorted_cols = sorted(data_cols, key=sort_key)
final_cols = ['record_id', 'redcap_event_name'] + sorted_cols
output_df = output_df[final_cols]

print(f"Output DataFrame shape: {output_df.shape}")
print(f"Columns: {len(output_df.columns)}")
output_df.head()


In [ ]:
output_df.dtypes

## Save Output


In [ ]:
# Save to CSV
output_path = FINAL_DIR / mapping_config['output']['filename']
output_df.to_csv(output_path, index=False)

print(f"Saved daily assessment upload to: {output_path}")
print(f"Shape: {output_df.shape}")
print(f"Patients: {len(output_df)}")
print(f"Columns: {len(output_df.columns)}")

# Summary
print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Generated at: {datetime.now().isoformat()}")
print(f"Days covered: {mapping_config['days_to_process']}")
print(f"Output file: {output_path}")
